# Complete Fine-tuning Pipeline

This notebook runs the complete workflow from dataset creation to model export.

**Data Flow:**
- Augmented data (with built-in filtering) → `data/generated/augmented_dataset.json`
- Converted data → `data/processed/converted_dataset.json`
- Train/Val splits → `data/processed/train.json` and `data/processed/val.json`


In [ ]:
import sys
import os
from pathlib import Path
import json

# Add project root to path
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Added to path: {project_root / 'src'}")

from datasets import Dataset
_original_map = Dataset.map
def _patched_map(self, *args, **kwargs):
    kwargs['num_proc'] = None
    return _original_map(self, *args, **kwargs)
Dataset.map = _patched_map

from src.utils import setup_logging, load_config, ensure_dir, get_project_root
from src.dataset_creation import load_initial_dataset, augment_dataset, save_dataset, DEFAULT_TOOL_SCHEMA, inject_tool_schema_into_dataset
from src.dataset_preparation import convert_to_unsloth_format, split_dataset, save_processed_dataset
from src.training import load_model, load_tokenizer_only, setup_lora, train_model, save_checkpoint
from src.export import export_all

import importlib
import src.evaluation
importlib.reload(src.evaluation)
from src.evaluation import load_saved_model, evaluate_on_dataset, generate_text, interactive_test, print_evaluation_results

setup_logging()

In [ ]:
# Load configuration
config = load_config()
print("Configuration loaded:")
print(f"  Model: {config['model']['name']}")
print(f"  Attention Backend: {config['attention_backend']}")
print(f"  Quantization: 16bit={config['quantization']['load_in_16bit']}")

## Step 1: Load Initial Dataset

Load your initial dataset from JSON file.

**Required format:** Each example must have a `messages` key:
```json
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

In [ ]:
# Resolve dataset path and load initial dataset
resolved_path = project_root / config['dataset']['raw_path'] / 'initial_dataset.json'
print(f"Loading dataset from: {resolved_path}")

if not resolved_path.exists():
    raise FileNotFoundError(f"Dataset file not found at: {resolved_path}")

# Inject tool schema so each example has AVAILABLE TOOLS (matches production LlamaService)
initial_data = load_initial_dataset(resolved_path)
print(f"✓ Loaded {len(initial_data)} initial examples (with tool schema injected)")

## Step 2: Load Tokenizer (for Dataset Formatting)

Load the tokenizer to format datasets according to the model's native chat template.
This is lightweight and doesn't load the full model.

In [ ]:
# Load tokenizer for dataset formatting
# tokenizer = load_tokenizer_only(
#     config['model']['name'],
#     config
# )
# print("✓ Tokenizer loaded for dataset formatting")

## Step 3: Augmentation (Optional)

Augmentation includes **built-in quality filtering** (length + deduplication); no separate filtering step.

Choose one:
- **Option A**: Run augmentation and continue with live data
- **Option B**: Load previously augmented data
- **Option C**: Skip augmentation entirely

In [ ]:
# OPTION A: Run augmentation (uncomment to use)
#
# Augmentation Strategies:
# - 'paraphrase': Rewrites user messages with different wording (same meaning)
# - 'expand': Generates new detailed assistant responses
# - 'variation': Creates new user messages asking similar things differently
# - 'response_variation': Generates alternative assistant responses with different styles
#
# Filtering (min_length, max_length, remove_duplicates) is built in; output is already filtered.

# augmented_data = augment_dataset(
#     initial_data,
#     api_url=config['lm_studio']['api_url'],
#     augmentation_strategy='variation',
#     num_augmentations_per_example=5,
#     save_path=project_root / 'data/generated/augmented_dataset.json',
#     save_format='json',
#     tool_schema=DEFAULT_TOOL_SCHEMA,  # Inject tool details (matches LlamaService)
#     tokenizer=tokenizer,  # Required: run Step 2 (Load Tokenizer) first
# )
# filtered_data = augmented_data  # Ready for Step 4 (conversion)
# print(f"✓ Augmented and saved: {len(augmented_data)} unique valid examples")

In [ ]:
# OPTION B: Load previously augmented data (default - run this to use saved augmented data)
# augmented_path = project_root / 'data/generated/augmented_dataset.json'
# if augmented_path.exists():
#     with open(augmented_path, 'r', encoding='utf-8') as f:
#         augmented_data = json.load(f)
#     # If data was saved without tools, inject: augmented_data = inject_tool_schema_into_dataset(augmented_data, DEFAULT_TOOL_SCHEMA)
#     filtered_data = augmented_data  # Already filtered; ready for Step 4 (conversion)
#     print(f"✓ Loaded augmented data: {len(augmented_data)} examples (filtered_data set for pipeline)")
# else:
#     print(f"⚠️ Augmented data not found at {augmented_path}")
#     augmented_data = initial_data  # Fallback
#     filtered_data = augmented_data

In [ ]:
# OPTION C: Skip augmentation (optional - run this cell to use initial_data without augmentation)
# augmented_data = initial_data
# filtered_data = augmented_data
# print(f"✓ Using initial data (no augmentation): {len(filtered_data)} examples")

## Step 4: Dataset Conversion

Convert dataset to Unsloth-compatible format using the tokenizer's chat template.

Choose one:
- **Option A**: Run conversion and continue with live data
- **Option B**: Load previously converted data

In [ ]:
# OPTION A: Run conversion (default)
# converted_data = convert_to_unsloth_format(
#     filtered_data,
#     tokenizer=tokenizer,
#     save_path=project_root / 'data/processed/converted_dataset.json',
#     save_format='json'
# )
# print(f"✓ Converted and saved: {len(converted_data)} examples")

In [ ]:
# OPTION B: Load previously converted data (uncomment to use)
# converted_path = project_root / 'data/processed/converted_dataset.json'
# if converted_path.exists():
#     with open(converted_path, 'r', encoding='utf-8') as f:
#         converted_data = json.load(f)
#     print(f"✓ Loaded converted data: {len(converted_data)} examples")
# else:
#     print(f"⚠️ Converted data not found at {converted_path}")
#     # Run Option A above first, or load tokenizer and run convert_to_unsloth_format(filtered_data, tokenizer=tokenizer, ...)

## Step 5: Train/Val Split

Choose one:
- **Option A**: Run split and continue with live data
- **Option B**: Load previously split data

In [ ]:
# OPTION A: Run split (default)
# train_data, val_data = split_dataset(
#     converted_data,
#     train_ratio=config['dataset']['train_ratio'],
#     val_ratio=config['dataset']['val_ratio'],
#     shuffle=True,
#     seed=42,
#     save_dir=project_root / 'data/processed',
#     save_format='json'
# )
# print(f"✓ Split and saved: Train={len(train_data)}, Val={len(val_data)}")

In [ ]:
# OPTION B: Load previously split data (uncomment to use)
train_path = project_root / 'data/processed/train.json'
val_path = project_root / 'data/processed/val.json'
if train_path.exists() and val_path.exists():
    with open(train_path, 'r', encoding='utf-8') as f:
        train_data = json.load(f)
    with open(val_path, 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    print(f"✓ Loaded split data: Train={len(train_data)}, Val={len(val_data)}")
else:
    print(f"⚠️ Split data not found. Run Option A above first.")
    train_data, val_data = [], []

In [ ]:
# Convert to HuggingFace Dataset format for training (run after Option A or B)
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data) if val_data else None
print(f"✓ Created HuggingFace datasets for training")

## Step 6: Model Loading & Setup

Load the base model and configure LoRA adapters.

In [ ]:
# Load model
model, tokenizer = load_model(
    config['model']['name'],
    config,
    max_seq_length=config['training']['max_seq_length']
)
print("✓ Model loaded successfully")

In [ ]:
# Setup LoRA
model = setup_lora(model, config, max_seq_length=config['training']['max_seq_length'])
print("✓ LoRA adapters configured")

## Step 7: Training

Train the model with your prepared dataset.

### Quick Start Guide

**For New Training (Option A):**
1. Run Steps 1-5 (Setup, Config, Dataset)
2. Run Step 6 (Load Model + Setup LoRA)
3. Run Option A cell below

**For Resuming Training (Option B):**
1. Run Steps 1-2 (Setup, Config, Imports) - *skip dataset steps*
2. Run Steps 3-5 (Load your datasets)
3. Run Step 6 (Load Model + Setup LoRA) - **MUST DO THIS!**
4. Run Helper cell to see checkpoints
5. Run Option B cell below

**Choose one option:**
- **Option A**: Train from scratch (new training run)
- **Option B**: Resume from a checkpoint (continue previous training)

In [ ]:
# Helper: List available checkpoints (run before Option B to see which checkpoint to resume from)
# output_dir = project_root / config['output']['base_dir'] / 'training'
# checkpoints = sorted(output_dir.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]) if p.name.split('-')[1].isdigit() else 0)
# if checkpoints:
#     print(f"Available checkpoints in {output_dir}:")
#     for cp in checkpoints:
#         print(f"  - {cp.name}")
#     print(f"\nLatest: {checkpoints[-1].name}")
# else:
#     print(f"No checkpoints found in {output_dir}. Run Option A first.")

In [ ]:
# OPTION A: Train from scratch (default)
output_dir = project_root / config['output']['base_dir'] / 'training'
resume_from_checkpoint = None

trainer = train_model(
    model,
    tokenizer,
    train_dataset,
    val_dataset,
    config,
    output_dir,
    resume_from_checkpoint=None
)
print(f"✓ Training complete! Checkpoints saved to: {output_dir}")

In [ ]:
# OPTION B: Resume from checkpoint (uncomment to use)
# output_dir = project_root / config['output']['base_dir'] / 'training'
# checkpoints = sorted(output_dir.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]) if p.name.split('-')[1].isdigit() else 0)
# if checkpoints:
#     resume_from_checkpoint = str(checkpoints[-1])
#     print(f"Resuming from: {resume_from_checkpoint}")
#     trainer = train_model(
#         model,
#         tokenizer,
#         train_dataset,
#         val_dataset,
#         config,
#         output_dir,
#         resume_from_checkpoint=resume_from_checkpoint
#     )
#     print(f"✓ Training resumed and complete! Checkpoints: {output_dir}")
# else:
#     print("⚠️ No checkpoints found. Run Option A first or set resume_from_checkpoint manually.")

## Step 8: Export

Export the trained model in all enabled formats.

In [ ]:
# Export in all enabled formats
export_dir = project_root / config['output']['base_dir'] / 'exports'
export_all(
    model,
    tokenizer,
    export_dir,
    config,
    model_name="fine_tuned_model"
)
print(f"\n✓ All exports saved to: {export_dir}")

## Step 9: Evaluation/Testing

Test the saved model by loading it in one of the exported formats.

**Choose which model format to load:**
- **Option A**: Load LoRA adapter (requires base model + adapter)
- **Option B**: Load merged 16-bit model (recommended)
- **Option C**: Load GGUF model (requires llama-cpp-python, works seamlessly with llama.cpp)

**All evaluation functions work with all model formats!**
- `evaluate_on_dataset()`: Auto-detects model type
- `generate_text()`: Works with HuggingFace and GGUF models
- `interactive_test()`: Shows model type and works with all formats

In [ ]:
# OPTION A: Load LoRA adapter (uncomment to use)
# export_dir = project_root / config['output']['base_dir'] / 'exports'
# eval_model, eval_tokenizer = load_saved_model(
#     model_path=export_dir / "lora_adapter",
#     model_format="lora",
#     base_model_name=config['model']['name'],
#     config=config
# )
# print(f"✓ Loaded LoRA adapter from: {export_dir / 'lora_adapter'}")

In [ ]:
# OPTION B: Load merged 16-bit model (recommended, uncomment to use)
# export_dir = project_root / config['output']['base_dir'] / 'exports'
# eval_model, eval_tokenizer = load_saved_model(
#     model_path=export_dir / "merged_16bit",
#     model_format="merged",
#     config=config
# )
# print(f"✓ Loaded merged model from: {export_dir / 'merged_16bit'}")

In [ ]:
# OPTION C: Load GGUF model (uncomment to use)
# Note: Requires llama-cpp-python: pip install llama-cpp-python
export_dir = project_root / config['output']['base_dir'] / 'exports'
gguf_path = export_dir / "gguf" / "model-fp16.gguf"
eval_model, eval_tokenizer = load_saved_model(
    model_path=gguf_path,
    model_format="gguf",
    config=config
)
print(f"✓ Loaded GGUF model from: {gguf_path}")

### Evaluate on Validation Dataset

Test the model on validation samples to see how it performs.
**Note:** Evaluation works with all model formats (LoRA, merged, GGUF).
Make sure you've loaded a model using one of the options above first!


In [ ]:
# # Evaluate on filtered dataset
# # IMPORTANT: First load a model using Option A, B, or C above!
# #
# # Convert filtered_data to Dataset format for evaluation
# eval_dataset = train_dataset
eval_dataset = val_dataset

print("Evaluating model on filtered data...\n")

eval_results = evaluate_on_dataset(
    eval_model,
    eval_tokenizer,
    eval_dataset,
    num_samples=5,  # Test on 5 samples
    max_new_tokens=200,
    config=config
)

print(f"\n✓ Evaluated {len(eval_results)} samples")

# Print results showing expected vs actual
print_evaluation_results(eval_results, num_to_print=5)

### Single Prompt Test

Test with a custom prompt. Works with all model formats (LoRA, merged, GGUF).

In [ ]:
# Test with custom prompt (using messages format)
# test_messages = [{"role": "user", "content": "What is machine learning?"}]

# print(f"User: {test_messages[0]['content']}\n")
# print("Generating response...\n")

# response = generate_text(
#     eval_model,
#     eval_tokenizer,
#     test_messages,
#     max_new_tokens=200,
#     temperature=0.7,
#     config=config
# )

# print("="*70)
# print("Assistant:")
# print(response)
# print("="*70)

### Interactive Testing Mode (Optional)

Enter prompts interactively to test the model. Type 'quit' to exit.

Works with all model formats (LoRA, merged, GGUF). The model type will be displayed.

In [ ]:
# Uncomment to enable interactive testing
# interactive_test(eval_model, eval_tokenizer, config=config)